In [1]:
import joblib
import pandas as pd

# Only what the test-file cleaning needs
for n in ['train_df', 'val_df', 'y_train']:
    globals()[n] = joblib.load(f'data/checkpoints/{n}.joblib')

# Check: expect train_df (59634, 10), val_df (14907, 10) and 28 classes
print(train_df.shape, val_df.shape, y_train.nunique())

(59634, 10) (14907, 10) 28


In [2]:
import html
import pandas as pd

# ------------------------------------------------------------------
# Fixed recipe, copied from Cleaning.ipynb. Nothing here is learned from the test file.
# ------------------------------------------------------------------
fix_map = {   # Cell 15: values cut off at the start
    'ibromyalgia': 'Fibromyalgia',
    'mance Anxiety': 'Performance Anxiety',
    'emale Infertility': 'Female Infertility',
    'atigue': 'Fatigue',
    'acial Wrinkles': 'Facial Wrinkles',
    'zen Shoulde': 'Frozen Shoulder',
    'lic Acid Deficiency': 'Folic Acid Deficiency',
    'amilial Mediterranean Feve': 'Familial Mediterranean Fever',
    'cal Segmental Glomerulosclerosis': 'Focal Segmental Glomerulosclerosis',
    'actor IX Deficiency': 'Factor IX Deficiency',
    'ailure to Thrive': 'Failure to Thrive',
    'amilial Cold Autoinflammatory Syndrome': 'Familial Cold Autoinflammatory Syndrome',
    'ungal Pneumonia': 'Fungal Pneumonia',
    'ungal Infection Prophylaxis': 'Fungal Infection Prophylaxis',
    'ibrocystic Breast Disease': 'Fibrocystic Breast Disease',
    'llicular Lymphoma': 'Follicular Lymphoma',
    'm Pain Disorder': 'Somatoform Pain Disorder',
    'unctional Gastric Disorder': 'Functional Gastric Disorder',
    'acial Lipoatrophy': 'Facial Lipoatrophy',
}
fix_map2 = {  # Cell 37: values cut off at the end
    'Overactive Bladde': 'Overactive Bladder',
    'Not Listed / Othe': 'Not Listed / Other',
    'Breast Cance': 'Breast Cancer',
    'Prostate Cance': 'Prostate Cancer',
    'Vertig': 'Vertigo',
    'Glioblastoma Multi': 'Glioblastoma Multiforme',
    'Dermatitis Herpeti': 'Dermatitis Herpetiformis',
    'Stomach Ulce': 'Stomach Ulcer',
    'Pain/Feve': 'Pain/Fever',
}
merge_map = {'Major Depressive Disorder': 'Depression', 'Chronic Pain': 'Pain'}

# The 30 classes come from TRAIN (not recomputed from the test file)
class_set = set(train_df['condition'].unique())
assert len(class_set) == 30, len(class_set)

# ------------------------------------------------------------------
# Replay, in the same order as the cleaning notebook
# ------------------------------------------------------------------
test_raw = pd.read_csv('data/drugsComTest_raw.csv')
print('Raw test file:', test_raw.shape)
print(test_raw.columns.tolist())
assert {'drugName', 'condition', 'review', 'date'} <= set(test_raw.columns), 'a needed column is missing'
work = test_raw.copy()
log, setaside = [], {}

def set_aside(step, mask):
    """Move the rows where mask is True into their own table, and log the step."""
    global work
    before = len(work)
    setaside[step] = work[mask].copy()
    work = work[~mask].reset_index(drop=True)
    assert len(work) + len(setaside[step]) == before     # nothing lost
    log.append({'step': step, 'rows_before': before, 'rows_after': len(work), 'rows_removed': before - len(work)})

key_cols = ['review_clean', 'condition', 'date']

# 1. Missing condition
set_aside('1. Missing condition', work['condition'].isna())

# 2. </span> junk in condition
set_aside('2. </span> junk', work['condition'].str.contains('</span>', regex=False))

# 3. "...Disorde" endings (this also covers Bipolar Disorde)
work['condition'] = work['condition'].str.replace('Disorde$', 'Disorder', regex=True)

# 4. Cut-off starts (fixed dictionary)
work['condition'] = work['condition'].replace(fix_map)

# 5. Leftover cut-off values (start with lowercase, except von Willebrand's Disease)
starts_lower = work['condition'].str.match(r'^[a-z]')
is_von_w = work['condition'] == "von Willebrand's Disease"
set_aside('3. Leftover cut-off values', starts_lower & ~is_von_w)

# 6. review_clean: unescape HTML, collapse whitespace, strip
assert work['review'].notna().all(), 'some reviews are missing'
work['review_clean'] = work['review'].apply(html.unescape)
work['review_clean'] = work['review_clean'].str.replace(r'\s+', ' ', regex=True)
work['review_clean'] = work['review_clean'].str.strip()

# 7. Weight Loss + Obesity -> Weight Management
work['condition'] = work['condition'].replace({'Weight Loss': 'Weight Management', 'Obesity': 'Weight Management'})

# 8. Cut-off ends (fixed dictionary)
work['condition'] = work['condition'].replace(fix_map2)

# 9. Duplicates on review_clean + condition + date (first copy kept)
set_aside('4. Duplicates', work.duplicated(subset=key_cols, keep='first'))

# 10. "Pe" -> Performance Anxiety, then the new duplicates it creates
work['condition'] = work['condition'].replace({'Pe': 'Performance Anxiety'})
set_aside('5. New duplicates after Pe repair', work.duplicated(subset=key_cols, keep='first'))

# 11. Long texts (10+ words) under 2+ conditions
conds_per_review = work.groupby('review_clean')['condition'].nunique()
multi = conds_per_review[conds_per_review > 1]
n_words = work['review_clean'].str.split().str.len()
set_aside('6. Multi-condition long texts', work['review_clean'].isin(multi.index) & (n_words >= 10))

# 12. Keep only the 30 classes (the list comes from train)
set_aside('7. Outside the 30 classes', ~work['condition'].isin(class_set))

# 13. Reviews under 5 words
n_words = work['review_clean'].str.split().str.len()
set_aside('8. Reviews under 5 words', n_words < 5)

# ------------------------------------------------------------------
# Steps taken from the preprocessing notebook
# ------------------------------------------------------------------
# 14. The two merges, in a new column (the original condition is kept)
work['condition_merged'] = work['condition'].replace(merge_map)

# 15. Exact duplicates on text + merged label (first copy kept), like train
set_aside('9. Duplicates after merges', work.duplicated(subset=['review_clean', 'condition_merged'], keep='first'))

# 16. Reviews whose text also appears in train
set_aside('10. Text also in train', work['review_clean'].isin(train_df['review_clean']))

test_df = work

# ------------------------------------------------------------------
# Row log and checks (nothing is saved here)
# ------------------------------------------------------------------
row_log = pd.DataFrame(log)
assert row_log['rows_removed'].sum() + len(test_df) == len(test_raw)     # everything adds up
display(row_log)
print('Raw test rows:', len(test_raw), '| Final test rows:', len(test_df))
print('Classes in the final test set:', test_df['condition_merged'].nunique())

Raw test file: (53766, 7)
['uniqueID', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount']


,step,rows_before,rows_after,rows_removed
0,1. Missing condition,53766,53471,295
1,2. </span> junk,53471,53200,271
2,3. Leftover cut-off values,53200,53112,88
3,4. Duplicates,53112,47777,5335
4,5. New duplicates after Pe repair,47777,47776,1
5,6. Multi-condition long texts,47776,47763,13
6,7. Outside the 30 classes,47763,32607,15156
7,8. Reviews under 5 words,32607,32337,270
8,9. Duplicates after merges,32337,32336,1
9,10. Text also in train,32336,14608,17728


Raw test rows: 53766 | Final test rows: 14608
Classes in the final test set: 28


In [3]:
# Check 1: the test labels must match the training classes exactly
print('Test labels not in train:', set(test_df['condition_merged']) - set(y_train))     # expect set()
print('Train classes missing in test:', set(y_train) - set(test_df['condition_merged']))  # expect set()

# Check 2: rows per class, smallest first (to see how thin the small classes are)
test_df['condition_merged'].value_counts().sort_values().head(8)

Test labels not in train: set()
Train classes missing in test: set()


condition_merged
Erectile Dysfunction            123
Osteoarthritis                  148
Muscle Spasm                    192
Bowel Preparation               193
Panic Disorder                  196
Generalized Anxiety Disorder    200
Constipation                    204
Bacterial Infection             211
Name: count, dtype: int64

In [4]:
# Check 3: possible unrepaired cut-offs among the rows dropped as "outside the 30 classes"
# (a dropped value that looks like a shortened version of one of your classes; nothing is changed)
out = setaside['7. Outside the 30 classes']['condition'].value_counts()
found = []
for value, n in out.items():
    for cls in class_set:
        extra = len(cls) - len(value)
        if 0 < extra <= 6 and (cls.startswith(value) or cls.endswith(value)):
            found.append({'value_in_test': value, 'rows': n, 'looks_like': cls})
print(pd.DataFrame(found) if found else 'None found')

# Information only: overlap with validation, unseen drugs, share of rows kept
print('Test reviews also in validation:', test_df['review_clean'].isin(val_df['review_clean']).sum())
print('Test rows with a drug unseen in train:', (~test_df['drugName'].isin(train_df['drugName'])).sum())
print('Share of raw test rows kept:', round(len(test_df) / len(test_raw), 3), '(train kept about 0.46)')

None found
Test reviews also in validation: 4372
Test rows with a drug unseen in train: 58
Share of raw test rows kept: 0.272 (train kept about 0.46)


In [5]:
# Are the removed test rows really the same reviews as in train, under another drug name?
removed = setaside['10. Text also in train']
train_first = train_df.drop_duplicates('review_clean')[['review_clean', 'drugName', 'condition', 'date']]
cmp_df = removed[['review_clean', 'drugName', 'condition', 'date']].merge(
    train_first, on='review_clean', suffixes=('_test', '_train'))

print('Rows compared:', len(cmp_df), '| removed rows:', len(removed))   # should match
print('Same condition:     ', round((cmp_df['condition_test'] == cmp_df['condition_train']).mean(), 3))
print('Same date:          ', round((cmp_df['date_test'] == cmp_df['date_train']).mean(), 3))
print('Different drug name:', round((cmp_df['drugName_test'] != cmp_df['drugName_train']).mean(), 3))

Rows compared: 17728 | removed rows: 17728
Same condition:      1.0
Same date:           1.0
Different drug name: 1.0


In [6]:
in_val = test_df['review_clean'].isin(val_df['review_clean'])
cls_table = pd.DataFrame({
    'test_rows': test_df['condition_merged'].value_counts(),
    'also_in_val': test_df[in_val]['condition_merged'].value_counts(),
}).fillna(0).astype(int)
cls_table['left_if_removed'] = cls_table['test_rows'] - cls_table['also_in_val']
cls_table.sort_values('left_if_removed')

,test_rows,also_in_val,left_if_removed
condition_merged,,,
Erectile Dysfunction,123,52,71
Osteoarthritis,148,48,100
Bowel Preparation,193,86,107
Abnormal Uterine Bleeding,212,100,112
Emergency Contraception,250,112,138
Constipation,204,58,146
Panic Disorder,196,49,147
Muscle Spasm,192,32,160
Generalized Anxiety Disorder,200,34,166


In [7]:
# Remove test rows whose text also appears in validation (run ONCE)
in_val = test_df['review_clean'].isin(val_df['review_clean'])
before = len(test_df)
setaside['11. Text also in validation'] = test_df[in_val].copy()
test_df = test_df[~in_val].reset_index(drop=True)
assert len(test_df) + len(setaside['11. Text also in validation']) == before

# Add the step to the row log
row_log = pd.concat([row_log, pd.DataFrame([{
    'step': '11. Text also in validation', 'rows_before': before,
    'rows_after': len(test_df), 'rows_removed': before - len(test_df)}])], ignore_index=True)

# Checks: expect 10,236 rows, 28 classes, no overlap with train or validation, and the log adds up
print(test_df.shape, test_df['condition_merged'].nunique())
print('In train:', test_df['review_clean'].isin(train_df['review_clean']).sum(),
      '| in validation:', test_df['review_clean'].isin(val_df['review_clean']).sum())
assert row_log['rows_removed'].sum() + len(test_df) == len(test_raw)
display(row_log)

(10236, 9) 28
In train: 0 | in validation: 0


,step,rows_before,rows_after,rows_removed
0,1. Missing condition,53766,53471,295
1,2. </span> junk,53471,53200,271
2,3. Leftover cut-off values,53200,53112,88
3,4. Duplicates,53112,47777,5335
4,5. New duplicates after Pe repair,47777,47776,1
5,6. Multi-condition long texts,47776,47763,13
6,7. Outside the 30 classes,47763,32607,15156
7,8. Reviews under 5 words,32607,32337,270
8,9. Duplicates after merges,32337,32336,1
9,10. Text also in train,32336,14608,17728


In [8]:
import os, joblib
os.makedirs('data/checkpoints', exist_ok=True)
test_df.to_csv('data/cleaned_test.csv', index=False)          # overwrites if run again
row_log.to_csv('data/test_cleaning_row_log.csv', index=False)
joblib.dump(test_df, 'data/checkpoints/test_df.joblib')
print('Saved', test_df.shape)

Saved (10236, 9)


In [9]:
import joblib
from scipy.sparse import hstack
from sklearn.metrics import f1_score, classification_report

# Load the frozen final pipeline (text + drug name, logistic regression)
for n in ['vectorizer', 'encoder', 'model_both']:
    globals()[n] = joblib.load(f'data/checkpoints/{n}.joblib')
DRUG_COL = 'drugName'

# TRANSFORM only, never refit
X_test = vectorizer.transform(test_df['review_clean'])
drug_test = encoder.transform(test_df[[DRUG_COL]])
X_test_both = hstack([X_test, drug_test]).tocsr()
y_test = test_df['condition_merged']

# Checks: columns must match training (34,186 + 1,160 = 35,346), and the labels must be known to the model
print('X_test_both:', X_test_both.shape)
print('Labels the model does not know:', set(y_test) - set(model_both.classes_))   # expect set()

y_pred_test = model_both.predict(X_test_both)
print('FINAL TEST macro F1 (text + drug):', round(f1_score(y_test, y_pred_test, average='macro'), 3))

X_test_both: (10236, 35346)
Labels the model does not know: set()
FINAL TEST macro F1 (text + drug): 0.855


In [10]:
# Per-class results, weakest first
report = pd.DataFrame(classification_report(y_test, y_pred_test, output_dict=True, zero_division=0)).T
report.drop(['accuracy', 'macro avg', 'weighted avg']).sort_values('f1-score').round(3)

,precision,recall,f1-score,support
Generalized Anxiety Disorder,0.757,0.319,0.449,166.0
Panic Disorder,0.616,0.361,0.455,147.0
Anxiety and Stress,0.551,0.502,0.526,235.0
Abnormal Uterine Bleeding,0.787,0.429,0.555,112.0
Fibromyalgia,0.874,0.615,0.722,169.0
Depression,0.689,0.804,0.742,929.0
Anxiety,0.737,0.772,0.754,741.0
Osteoarthritis,0.958,0.680,0.795,100.0
Bipolar Disorder,0.846,0.813,0.829,385.0
Migraine Prevention,0.870,0.890,0.880,218.0


In [11]:
   import os
   for f in ['data/cleaned_test.csv', 'data/test_cleaning_row_log.csv', 'data/checkpoints/test_df.joblib']:
       print(f, os.path.exists(f))

data/cleaned_test.csv True
data/test_cleaning_row_log.csv True
data/checkpoints/test_df.joblib True
